# 04 — Cross-species conservation, and the controls that constrain it

**Claims under test**
1. Cross-species positional agreement is real and threshold-robust. **Three fold values coexist and
   must not be conflated:** pooled estimator 2.90 (universe 13,867), stratified estimator **3.39**
   (stratification *strengthens* it), strict-1:1 homology comparison 3.11 (universe 9,550).
   Coordinate-independence: 2.90 on isodepth vs 2.91 on pseudotime, same estimator.
2. It is **not decidualization-specific in aggregate** — uninduced 2.581 vs induced 2.377.
3. But **37 genes gain** conserved positional information on decidualization, and 31 lose it
   (enclosing set 2.33x, p=1.5e-24).

Claim 2 is a negative result that could have killed the story. It is what makes claim 3 credible
rather than assumed — report it, don't bury it.

In [ ]:
# --- paths & helpers (edit ROOT if you move things) ---------------------------
import os, json, glob, pathlib, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")

X    = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/xmens_final")
REV  = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/decidualization/revisions")
NET  = pathlib.Path("/n/netscratch/eddy_lab/Everyone/nhilgert")
GP   = REV/"crossspecies_radial_gp"
DELIV= X/"data/deliverables"
ISO  = X/"results/isodepth"
MIS  = X/"results/mi_stratified"
DATA = pathlib.Path("data")     # staged artifacts, see README
ORTH = pathlib.Path("/n/lab_storage/eddy_lab/users/nhilgert/human_ivd_repro/orthogroups/ensembl_human_mouse_components.tsv")

HS_DELIV = DELIV/"2026_08_28_Cevrim_Human_Only_Fib_Dec_Cells.h5ad"
MM_DELIV = DELIV/"2026_08_28_Cevrim_XMens_Only_Fib_Dec_Cells.h5ad"   # NOTE: "XMens", not "Mouse"

_checks = []
def verify(label, got, expected, tol=0.02, note=""):
    """Recompute vs recorded. Prints PASS/MISMATCH and accumulates a summary."""
    try:
        ok = (abs(float(got) - float(expected)) <= tol * max(1.0, abs(float(expected)))
              if expected is not None else None)
    except (TypeError, ValueError):
        ok = (got == expected)
    tag = "PASS " if ok else ("MISMATCH" if ok is False else "n/a  ")
    _checks.append((label, got, expected, tag))
    print(f"[{tag}] {label}: recomputed={got}  recorded={expected}" + (f"   ({note})" if note else ""))
    return ok

def check_summary():
    d = pd.DataFrame(_checks, columns=["claim","recomputed","recorded","status"])
    n_bad = int((d.status.str.strip()=="MISMATCH").sum())
    print(f"\n{len(d)} checks | {n_bad} MISMATCH")
    return d

def ortholog_1to1():
    o = pd.read_csv(ORTH, sep="\t")
    o = o[o.homology_type=="ortholog_one2one"]
    piv = o.pivot_table(index="component_id", columns="species", values="gene_symbol",
                        aggfunc=lambda s: s.iloc[0] if len(s)==1 else np.nan).dropna()
    return piv[["human","mouse"]].drop_duplicates()

print("data dir:", DATA.resolve(), "| files:", len(list(DATA.glob("*"))) if DATA.exists() else "MISSING")
print("env ok |", "deliverables:", HS_DELIV.exists(), MM_DELIV.exists(), "| isodepth dir:", ISO.exists())


## 4.1 Recompute the cross-species overlap from the two MI tables

This rebuilds the headline enrichment end-to-end: strict 1:1 orthologs -> top-decile in each species -> hypergeometric.

In [ ]:
from scipy.stats import hypergeom
o2o = ortholog_1to1()
Sh = pd.read_csv(MIS/"human_isodepth_stratified.csv").set_index("gene")["mi_stratified"]
Sm = pd.read_csv(MIS/"mouse_isodepth_stratified.csv").set_index("gene")["mi_stratified"]
L = o2o[o2o.human.isin(Sh.index) & o2o.mouse.isin(Sm.index)].drop_duplicates()
N = len(L); k = int(round(N*0.10))
th = set(Sh.loc[L.human].sort_values(ascending=False).index[:k])
tm = set(Sm.loc[L.mouse].sort_values(ascending=False).index[:k])
hit_h = L.human.isin(th); hit_m = L.mouse.isin(tm)
ov = int((hit_h & hit_m).sum()); exp = int(hit_h.sum())*int(hit_m.sum())/N
p = float(hypergeom.sf(ov-1, N, int(hit_h.sum()), int(hit_m.sum())))
print(f"universe={N:,}  k(10%)={k}  overlap={ov}  expected={exp:.1f}  fold={ov/exp:.3f}  p={p:.3e}")
# THREE different fold values exist, from three different constructions. Do not conflate:
#   2.90 = POOLED MI estimator, universe 13,867   (mi_capstone.json)
#   3.39 = STRATIFIED estimator, this universe    (recomputed here; stratification strengthens it)
#   3.11 = strict-1:1 homology comparison, universe 9,550 (homology_model_comparison.json)
verify("cross-species fold enrichment (STRATIFIED estimator)", round(ov/exp,2), 3.39, tol=0.06,
       note=f"universe={N}; the pooled estimator on universe 13,867 gives 2.90 - different quantity")
verify("overlap is far above chance", bool(p < 1e-50), True)

## 4.2 Coordinate-independence: the same number on pseudotime

Recorded: isodepth fold 2.90 vs pseudotime fold 2.91 — the conservation conclusion does not depend on which axis you measure against.

In [ ]:
p_ = DATA/"mi_capstone.json"
if not p_.exists(): p_ = None
if p_:
    d = json.load(open(p_))
    for key, lab in (("xspecies_isodepth","isodepth"), ("xspecies_dpt","pseudotime")):
        v = d[key]
        print(f"{lab:11s} universe={v['universe']:,} overlap={v['overlap']} expected={v['expected']} fold={v['fold']} p={v['p']:.2e}")
    verify("isodepth fold", d["xspecies_isodepth"]["fold"], 2.90)
    verify("pseudotime fold", d["xspecies_dpt"]["fold"], 2.91)
    verify("the two axes agree to <0.05 fold",
           bool(abs(d["xspecies_isodepth"]["fold"]-d["xspecies_dpt"]["fold"]) < 0.05), True)
    print("\nwithin-species top-decile overlap, isodepth vs pseudotime:")
    print("  human:", d["human_dpt_vs_isodepth"])
else:
    print("mi_capstone.json not found")

## 4.3 The specificity control — the result that redirected the project

Design: uninduced mouse (4 blobs, 3,958 cells, **measured** boundary depth) vs induced subsampled to
matched n with 10 replicate draws, identical coordinate construction on both sides, shared gene
universe, per-arm permuted-coordinate null.

Recorded: **uninduced 2.581 vs induced 2.377 +/- 0.094.** No decidualization-specific advantage.

In [ ]:
p_ = DATA/"specificity_control.json"
if not p_.exists(): p_ = None
if p_:
    d = json.load(open(p_))
    print("design:", json.dumps(d.get("design", {}), indent=1)[:400])
    for k_, v in d.get("verdict", {}).items(): print(f"  {k_}: {v}")
    ua = d.get("arm_A_uninduced", d.get("uninduced", {}))
    ib = d.get("arm_B_induced", d.get("induced", {}))
    print("\nuninduced:", {k2:v2 for k2,v2 in ua.items() if not isinstance(v2,(list,dict))})
    print("induced  :", {k2:v2 for k2,v2 in ib.items() if not isinstance(v2,(list,dict))})
    print("\nINTERPRETATION: the conserved positional program is a property of endometrial stroma,")
    print("not of decidualization. Aggregate enrichment cannot distinguish them.")
else:
    print("specificity_control.json not found")

## 4.4 The gene-level rescue — what the aggregate test could not see

Sets, from `decidua_specific_conserved.csv`:

| set | definition | n |
|---|---|---|
| A | human top-10% AND induced top-10% | 221 |
| B | A minus genes already positional in uninduced | **147** (2.33x, p=1.5e-24) |
| C | B with gain_z > 2 | 115 |
| **D** | C where uninduced MI is at the noise floor | **37** |
| E | reverse: positional in uninduced, lost on induction | 31 |

**WNT5A is in set E (loss), not the gain set** — gain −0.0223, z −1.91. I asserted the opposite once;
the cell below checks it so the error cannot be repeated silently.

In [ ]:
p_ = DATA/"decidua_specific_conserved.csv"
if not p_.exists(): p_ = None
if p_:
    D = pd.read_csv(p_)
    print("columns:", list(D.columns))
    gz = next((c for c in D.columns if "gain_z" in c or c=="z"), None)
    gc = next((c for c in D.columns if c=="gain" or "gain" in c and "z" not in c), None)
    gene = next((c for c in D.columns if c.lower() in ("gene","mouse","human","symbol")), D.columns[0])
    CONV = ["Runx1","Cebpb","Cd63","Gja1","Nampt"]     # verified cross-method convergence
    DISP = ["Wnt5a"]                                    # the one I got WRONG
    for g in CONV + DISP:
        r = D[D[gene].astype(str).str.lower()==g.lower()]
        if len(r):
            row = r.iloc[0]
            print(f"  {g:8s} gain={row[gc]:+.4f}  z={row[gz]:+.2f}")
            if g in CONV: verify(f"{g} is a GAIN gene (z>2)", bool(row[gz] > 2), True)
            else:         verify(f"{g} is NOT a gain gene", bool(row[gz] > 2), False,
                                 note="WNT5A is on the LOSS side - do not list it as convergence")
else:
    print("decidua_specific_conserved.csv not found")

## Summary

In [ ]:
check_summary()